In [1]:
import os
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets

In [2]:
try:
    from torchinfo import summary
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "torchinfo"])
    from torchinfo import summary

try:
    import torchmetrics
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "torchmetrics"])
    import torchmetrics


In [3]:
repo_url = "https://raw.githubusercontent.com/vardanskamra/Deep-Multimedia-Steganography/main"
file = "helper_functions.py"
file_url = f"{repo_url}/{file}"
os.system(f"curl -s -f {file_url} -o {file}")
print("Downloaded all file successfully!")

Downloaded all file successfully!


In [4]:
from helper_functions import (train, 
    test,
    random_attack, 
    apply_attacks,
    plot_metrics,
    visualize_images)
from helper_functions import transform


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
dataset = datasets.Caltech256(root='./data', download=True, transform=transform)

# Using x% of the dataset
subset_size = int(1 * len(dataset))  
train_size = int(0.8 * subset_size)  
test_size = subset_size - train_size  

subset_dataset, _ = torch.utils.data.random_split(dataset, [subset_size, len(dataset) - subset_size])

train_dataset, test_dataset = torch.utils.data.random_split(subset_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=os.cpu_count(), drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, drop_last=True)

Files already downloaded and verified


In [7]:
print(f"Train Dataset Length: {len(train_dataset)}")
print(f"Test Dataset Length: {len(test_dataset)}")
print(f"Train DataLoader Length: {len(train_loader)}")
print(f"Test DataLoader Length: {len(test_loader)}")

sample, label = next(iter(train_loader))
print(f"Sample Shape: {sample.shape}")
print(f"Label Shape: {label.shape}")

Train Dataset Length: 24485
Test Dataset Length: 6122
Train DataLoader Length: 765
Test DataLoader Length: 191
Sample Shape: torch.Size([32, 3, 256, 256])
Label Shape: torch.Size([32])


In [8]:
class PrepNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Initial multi-scale processing
        self.initialP3 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialP5 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialP7 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU())

        # Upsampling and final processing
        self.upscale = nn.Sequential(
            nn.ConvTranspose2d(150, 150, kernel_size=4, stride=2, padding=1),  # 128->256
            nn.SiLU(),
            nn.Conv2d(150, 150, 3, padding=1),
            nn.SiLU()
        )

    def forward(self, p):
        p1 = self.initialP3(p)
        p2 = self.initialP5(p)
        p3 = self.initialP7(p)
        combined = torch.cat((p1, p2, p3), 1)
        return self.upscale(combined)



In [9]:
class HidingNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.initialH3 = nn.Sequential(
            nn.Conv2d(153, 50, kernel_size=3, padding=1),  
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialH5 = nn.Sequential(
            nn.Conv2d(153, 50, kernel_size=5, padding=2),  
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialH7 = nn.Sequential(
            nn.Conv2d(153, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50))

        # Final processing remains same
        self.finalH = nn.Sequential(
            nn.Conv2d(150, 3, kernel_size=1),
            nn.Sigmoid())

    def forward(self, cover, secret_prepared):
        h = torch.cat([cover, secret_prepared], dim=1)
        h1 = self.initialH3(h)
        h2 = self.initialH5(h)
        h3 = self.initialH7(h)
        combined = torch.cat((h1, h2, h3), 1)
        return self.finalH(combined)

In [10]:
class RevealNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Downsampling layers with stride=2
        self.initialR3 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=3, stride=2, padding=1),  # 256->128
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=3, padding=1),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialR5 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=5, stride=2, padding=2),  # 256->128
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=5, padding=2),
            nn.BatchNorm2d(50),
            nn.SiLU())
        
        self.initialR7 = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=7, stride=2, padding=3),  # 256->128
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU(),
            nn.Conv2d(50, 50, kernel_size=7, padding=3),
            nn.BatchNorm2d(50),
            nn.SiLU())

        # Final processing at 128x128 resolution
        self.finalR = nn.Sequential(
            nn.Conv2d(150, 50, kernel_size=1),
            nn.SiLU(),
            nn.Conv2d(50 ,3, kernel_size=1),
            nn.Sigmoid())

    def forward(self, r):
        r1 = self.initialR3(r)
        r2 = self.initialR5(r)
        r3 = self.initialR7(r)
        combined = torch.cat((r1, r2, r3), 1)
        return self.finalR(combined)

In [11]:
print(f"Device: {device}")
prep_net = nn.DataParallel(PrepNetwork()).to(device)
hide_net = nn.DataParallel(HidingNetwork()).to(device)
reveal_net = nn.DataParallel(RevealNetwork()).to(device)

Device: cuda


In [12]:
print("PrepNetwork Summary:")
print(summary(prep_net, input_size=(16, 3, 128, 128)))

cover = torch.randn(16, 3, 256, 256).to(device)
secret_prepared = torch.randn(16, 150, 256, 256).to(device)

print("\nHidingNetwork Summary:")
print(summary(hide_net, input_data=[cover, secret_prepared])) # 6 channels (cover + secret_prepared)

print("\nRevealNetwork Summary:")
print(summary(reveal_net, input_size=(16, 3, 256, 256)))

PrepNetwork Summary:
Layer (type:depth-idx)                   Output Shape              Param #
DataParallel                             [16, 150, 256, 256]       --
├─PrepNetwork: 1-1                       [8, 150, 256, 256]        1,407,500
├─PrepNetwork: 1-4                       --                        (recursive)
│    └─Sequential: 2-1                   [8, 50, 128, 128]         92,100
│    └─Sequential: 2-13                  --                        (recursive)
│    │    └─Conv2d: 3-1                  [8, 50, 128, 128]         1,400
├─PrepNetwork: 1-3                       [8, 150, 256, 256]        --
├─PrepNetwork: 1-4                       --                        (recursive)
│    └─Sequential: 2-3                   [8, 50, 128, 128]         --
│    └─Sequential: 2-13                  --                        (recursive)
│    │    └─Conv2d: 3-2                  [8, 50, 128, 128]         --
│    │    └─BatchNorm2d: 3-3             [8, 50, 128, 128]         --
│    │    └─Ba

In [13]:
optimizer_prep_hide = torch.optim.Adam(list(prep_net.parameters()) + list(hide_net.parameters()), lr=0.001)
optimizer_reveal = torch.optim.Adam(reveal_net.parameters(), lr=0.001)

In [14]:
# checkpoint = torch.load('/kaggle/working/model_checkpoint.pth', map_location=device)

In [ ]:
train_metrics = train(dataloader=train_loader,
                prep_net=prep_net,
                hide_net=hide_net,
                reveal_net=reveal_net,
                optimizer_prep_hide=optimizer_prep_hide,
                optimizer_reveal=optimizer_reveal,
                beta=1.0,
                prep_hide_max_norm=1.0,
                reveal_max_norm=1.5,
                attacks=False,
                epochs=5,
                checkpoint=None,
                device=device)

/usr/local/lib/python3.10/dist-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `PeakSignalNoiseRatio` from `torchmetrics` was deprecated and will be removed in 2.0. Import `PeakSignalNoiseRatio` from `torchmetrics.image` instead.
  _future_warning(
/usr/local/lib/python3.10/dist-packages/torchmetrics/utilities/prints.py:62: FutureWarning: Importing `StructuralSimilarityIndexMeasure` from `torchmetrics` was deprecated and will be removed in 2.0. Import `StructuralSimilarityIndexMeasure` from `torchmetrics.image` instead.
  _future_warning(


Epoch [1/5], Loss: 0.0191, PSNR: 22.0720, SSIM: 0.7025, NC: 0.9418, Pixel Loss (Cover-Stego): 0.0638, Pixel Loss (Secret-Revealed): 0.0745
Epoch [2/5], Loss: 0.0074, PSNR: 24.9994, SSIM: 0.7995, NC: 0.9816, Pixel Loss (Cover-Stego): 0.0445, Pixel Loss (Secret-Revealed): 0.0454
Epoch [3/5], Loss: 0.0053, PSNR: 27.0844, SSIM: 0.8243, NC: 0.9851, Pixel Loss (Cover-Stego): 0.0351, Pixel Loss (Secret-Revealed): 0.0405
Epoch [4/5], Loss: 0.0044, PSNR: 28.0527, SSIM: 0.8467, NC: 0.9876, Pixel Loss (Cover-Stego): 0.0313, Pixel Loss (Secret-Revealed): 0.0372


In [ ]:
plot_metrics(train_metrics)

In [ ]:
test_metrics = test(prep_net=prep_net,
               hide_net=hide_net,
               reveal_net=reveal_net,
               dataloader=test_loader,
               beta = 1.0,
               visualize = 2,
               attacks=None,                  
               device=device)

In [ ]:
torch.save(prep_net.state_dict(), "/kaggle/working/prep_net.pth")
torch.save(hide_net.state_dict(), "/kaggle/working/hide_net.pth")
torch.save(reveal_net.state_dict(), "/kaggle/working/reveal_net.pth")

In [ ]:
# 5-Epochs of Full Dataset training @ beta=1, NO ATTACKS
# 5-Epochs of Full Dataset training @ beta=0.9, NO ATTACKS
# 10-Epochs of Full Dataset training @ beta=0.9, WITH ATTACKS
# 30-Epochs of 10% Dataset training @ beta=0.8, WITH ATTACKS